# Pandas — Phase 4: Data Transformation & Feature Engineering
### Credit Card Risk Analysis Track

**Topics in this phase:**
16. Creating & Modifying Columns
17. Column Renaming & Dropping — `.rename()`, `.drop()`
18. Conditional Mapping — `np.where()`
19. Element-wise Mapping — `.map()` & `.replace()`
20. Sorting Data — `.sort_values()`

**Dataset:** back to the clean `loan_applications.csv` from Phase 1 (feature engineering doesn't need the Phase 3 messiness — but we do a quick median-impute in Setup so the ratio columns you build don't fill with NaN). Keep it in the same folder as this notebook.

**Note on this notebook:** many tasks below **modify `df` in place** (adding, renaming, or dropping columns), and later tasks depend on those changes — same pipeline structure as the numpy mini project. Run cells top to bottom.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual dataset before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("loan_applications.csv", parse_dates=["application_date"])
df["annual_income"] = df["annual_income"].fillna(df["annual_income"].median())
df["credit_score"] = df["credit_score"].fillna(df["credit_score"].median())
print(df.shape)
df.head()

## Topic 16: Creating & Modifying Columns

This is the heart of feature engineering — turning raw fields into signals a model can actually use.

**Q1.** Create a new column `debt_to_income_ratio` = `loan_amount / annual_income`. Print the first 3 values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["debt_to_income_ratio"] = df["loan_amount"] / df["annual_income"]
print(df["debt_to_income_ratio"].head(3).tolist())

**Q2.** Create `total_debt_ratio` = `(existing_debt + loan_amount) / annual_income` — a fuller picture than `debt_to_income_ratio` alone, since it accounts for debt the borrower already carries.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["total_debt_ratio"] = (df["existing_debt"] + df["loan_amount"]) / df["annual_income"]
print(df["total_debt_ratio"].head(3).tolist())

**Q3.** Create `monthly_payment_estimate` = `loan_amount / loan_term_months` — a naive estimate (ignoring interest) of what the borrower pays each month.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["monthly_payment_estimate"] = df["loan_amount"] / df["loan_term_months"]
print(df["monthly_payment_estimate"].head(3).tolist())

**Q4.** Create `income_per_dependent` = `annual_income / (num_dependents + 1)` (the `+ 1` accounts for the borrower themself, so it's never dividing by zero).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["income_per_dependent"] = df["annual_income"] / (df["num_dependents"] + 1)
print(df["income_per_dependent"].head(3).tolist())

**Q5.** Modify `loan_amount` **in place**, rounding every value to the nearest hundred (use `.round(-2)` — negative rounding digits round to the left of the decimal point).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["loan_amount"] = df["loan_amount"].round(-2)
print(df["loan_amount"].head(3).tolist())

**Q6.** Use `.assign()` to create **two columns in one call**: `debt_to_income_pct` (the ratio from Q1, times 100, rounded to 2 decimals) and `is_large_loan` (`True` if `loan_amount` exceeds 20000). `.assign()` takes keyword arguments where each value can be a lambda taking the DataFrame — this lets you chain column creation without repeating `df[...] = ` every time.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df = df.assign(
    debt_to_income_pct=lambda d: (d["debt_to_income_ratio"] * 100).round(2),
    is_large_loan=lambda d: d["loan_amount"] > 20000,
)
print(df[["debt_to_income_pct", "is_large_loan"]].head(3))

## Topic 17: Column Renaming & Dropping

A tidy, well-named DataFrame is much easier to hand off to a modeling step (or a teammate) later.

**Q7.** Rename `application_id` to `Applicant_ID` using `.rename(columns={...})`. Confirm the new name exists.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df = df.rename(columns={"application_id": "Applicant_ID"})
print("Applicant_ID" in df.columns)

**Q8.** Rename `annual_income`, `credit_score`, and `loan_amount` to `Annual_Income`, `Credit_Score`, and `Loan_Amount` respectively, in one `.rename()` call.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df = df.rename(columns={
    "annual_income": "Annual_Income",
    "credit_score": "Credit_Score",
    "loan_amount": "Loan_Amount",
})
print([c for c in df.columns if c[0].isupper()])

**Q9.** Drop `application_date` — useful for the raw ledger, but not something a model can use directly as a number. Confirm it's gone.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df = df.drop(columns=["application_date"])
print("application_date" in df.columns)

**Q10.** Drop `monthly_payment_estimate` and `income_per_dependent` together in one `.drop()` call — imagine you decided during feature review that these two weren't adding predictive value. Confirm both are gone.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df = df.drop(columns=["monthly_payment_estimate", "income_per_dependent"])
print("monthly_payment_estimate" in df.columns, "income_per_dependent" in df.columns)

**Q11.** Rather than renaming column-by-column, bulk-rename **every** column to lowercase in one line, by reassigning `df.columns` to a list comprehension (`c.lower() for c in df.columns`). Print the final column list.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df.columns = [c.lower() for c in df.columns]
print(df.columns.tolist())

## Topic 18: Conditional Mapping (`np.where`)

Most models want clean binary or small-integer flags, not raw continuous thresholds re-evaluated every time.

**Q12.** Create `high_risk_flag`: `1` where `credit_score < 600`, else `0`, using `np.where`. Print how many are flagged.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["high_risk_flag"] = np.where(df["credit_score"] < 600, 1, 0)
print(df["high_risk_flag"].sum())

**Q13.** Create `has_dependents`: `1` if `num_dependents > 0`, else `0`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["has_dependents"] = np.where(df["num_dependents"] > 0, 1, 0)
print(df["has_dependents"].sum())

**Q14.** Create `was_denied`: `1` where `loan_status == "Denied"`, else `0` — a stand-in for a "prior default" flag you'd build the same way if that column existed.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["was_denied"] = np.where(df["loan_status"] == "Denied", 1, 0)
print(df["was_denied"].sum())

**Q15.** `np.where` isn't limited to two outcomes if you nest it: create `risk_level` with 3 tiers — `0` if `credit_score >= 670`, `1` if `credit_score >= 580`, else `2` — by nesting a second `np.where` inside the "else" branch of the first. Print the value counts.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["risk_level"] = np.where(
    df["credit_score"] >= 670, 0, np.where(df["credit_score"] >= 580, 1, 2)
)
print(df["risk_level"].value_counts().sort_index())

**Q16.** Create `high_debt_flag`: `1` where **both** `debt_to_income_ratio > 0.4` **and** `credit_score < 650`, else `0` — combine the two conditions with `&` before passing them to `np.where`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["high_debt_flag"] = np.where(
    (df["debt_to_income_ratio"] > 0.4) & (df["credit_score"] < 650), 1, 0
)
print(df["high_debt_flag"].sum())

## Topic 19: Element-wise Mapping (`.map` & `.replace`)

`np.where` is for conditions on a value. `.map`/`.replace` are for a direct lookup table from one category to another.

**Q17.** Given `ownership_map = {"Rent": 0, "Mortgage": 1, "Own": 2}`, create `home_ownership_code` by mapping `home_ownership` through it with `.map()`. Print the first 5 values.

In [ ]:
ownership_map = {"Rent": 0, "Mortgage": 1, "Own": 2}

# YOUR CODE HERE


**Solution**

In [ ]:
ownership_map = {"Rent": 0, "Mortgage": 1, "Own": 2}
df["home_ownership_code"] = df["home_ownership"].map(ownership_map)
print(df["home_ownership_code"].head(5).tolist())

**Q18.** Do the same mapping again with `.replace(ownership_map)` instead of `.map()`, into `home_ownership_code_v2`. Confirm the **values** match Q17's result, and print both columns' dtypes — `.replace()` tends to be more conservative about dtype conversion than `.map()`, which is one reason `.map()` is usually preferred for this kind of clean full-column recode.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["home_ownership_code_v2"] = df["home_ownership"].replace(ownership_map)
print(
    (df["home_ownership_code_v2"] == df["home_ownership_code"]).all(),
    df["home_ownership_code"].dtype,
    df["home_ownership_code_v2"].dtype,
)

**Q19.** Given `employment_map = {"Employed": 0, "Self-Employed": 1, "Retired": 2}` — notice `"Unemployed"` is **not** in the map on purpose. Map `employment_status` through it into `employment_code`. Any unmapped category becomes `NaN` from `.map()` — fill those with `-1` using `.fillna(-1)`, then cast to `int`. Print the value counts.

In [ ]:
employment_map = {"Employed": 0, "Self-Employed": 1, "Retired": 2}

# YOUR CODE HERE


**Solution**

In [ ]:
employment_map = {"Employed": 0, "Self-Employed": 1, "Retired": 2}
df["employment_code"] = df["employment_status"].map(employment_map).fillna(-1).astype(int)
print(df["employment_code"].value_counts())

**Q20.** `.map()` also accepts a **function**, not just a dict. Create `age_group` by mapping `applicant_age` through a lambda: `"Young"` if under 35, `"Mid-Career"` if under 55, else `"Senior"`. Print the value counts.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df["age_group"] = df["applicant_age"].map(
    lambda age: "Young" if age < 35 else ("Mid-Career" if age < 55 else "Senior")
)
print(df["age_group"].value_counts())

**Q21.** Given `status_shorthand = {"Approved": "APR", "Denied": "DEN", "Pending": "PEN"}`, create `loan_status_code` using `.replace()` (this is the kind of partial/text-to-text substitution `.replace()` is actually well suited for). Print the unique values.

In [ ]:
status_shorthand = {"Approved": "APR", "Denied": "DEN", "Pending": "PEN"}

# YOUR CODE HERE


**Solution**

In [ ]:
status_shorthand = {"Approved": "APR", "Denied": "DEN", "Pending": "PEN"}
df["loan_status_code"] = df["loan_status"].replace(status_shorthand)
print(df["loan_status_code"].unique())

## Topic 20: Sorting Data

Sorting turns a flat table into a ranked list — the first step toward "who do we look at first?"

**Q22.** Sort `df` by `debt_to_income_ratio` from highest to lowest, into `sorted_by_dti_desc`. Print the top 3 ratios.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
sorted_by_dti_desc = df.sort_values("debt_to_income_ratio", ascending=False)
print(sorted_by_dti_desc[["debt_to_income_ratio"]].head(3))

**Q23.** Sort by **two** columns at once into `sorted_multi`: `credit_score` ascending (worst first), then `loan_amount` descending as a tiebreaker — pass a list to both `by=` and `ascending=`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
sorted_multi = df.sort_values(["credit_score", "loan_amount"], ascending=[True, False])
print(sorted_multi[["credit_score", "loan_amount"]].head(3))

**Q24.** Sort by `debt_to_income_ratio` descending again, but this time chain `.reset_index(drop=True)` so the row labels become a clean `0, 1, 2, ...` instead of carrying over the original (now-scrambled) index. Store as `sorted_reset` and print its first 5 index values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
sorted_reset = df.sort_values("debt_to_income_ratio", ascending=False).reset_index(drop=True)
print(sorted_reset.index[:5].tolist())

**Q25.** For "top N" style questions, sorting the whole DataFrame just to slice off the top is wasteful. Use `.nlargest(5, "debt_to_income_ratio")` directly into `top_5_riskiest_dti` instead. Print the ratios.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
top_5_riskiest_dti = df.nlargest(5, "debt_to_income_ratio")
print(top_5_riskiest_dti["debt_to_income_ratio"].tolist())

**Q26.** Similarly, use `.nsmallest(5, "annual_income")` to get the 5 lowest-income borrowers into `bottom_5_income`, without a full sort.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
bottom_5_income = df.nsmallest(5, "annual_income")
print(bottom_5_income["annual_income"].tolist())

## ✅ Checkpoint

**What you covered:**
- Creating & modifying columns: ratio/derived features, in-place rounding, and `.assign()` for creating multiple columns in one chainable call
- Renaming & dropping: single and bulk `.rename(columns=)`, `.drop(columns=)`, and reassigning `df.columns` directly for a full bulk rename
- Conditional mapping with `np.where`: binary flags, nested `np.where` for 3+ tiers, and combining boolean conditions before passing them in
- Element-wise mapping: `.map()` with a dict (and what happens to unmapped values), `.map()` with a function/lambda, `.replace()` and how it differs from `.map()` in dtype handling
- Sorting: single and multi-column `.sort_values()`, `ascending=` per column, `.reset_index(drop=True)` after sorting, and `.nlargest()`/`.nsmallest()` as a faster shortcut for top-N

**Why it matters for the project:** everything in this notebook — the ratios, the flags, the recoded categories — is exactly the kind of feature scikit-learn models actually consume. Raw columns like `home_ownership` (text) or `credit_score` (unbounded number) are far less useful to a model than `home_ownership_code` or `risk_level`.

**What's next:** Phase 5, whenever you're ready — let me know the topics you want covered.